# Links

## Imports

In [32]:
import pandas as pd
import csv, io

In [33]:
root = '../data'

## Read file

In [34]:
# read every column as strings first to inspect the raw format
links_raw = pd.read_csv(root + '/links.csv', dtype=str)
print(f"There are {links_raw.shape[0]} rows and {links_raw.shape[1]} columns in the links dataset")
display(links_raw.head())

# check that all values are digits and count the number of digits per column
digits = r'\d+'
for column_name in links_raw.columns:
    c = links_raw[column_name].dropna()
    print(f"{column_name}: {(~c.str.fullmatch(digits)).sum()} values that are not digits, number of digits: {c.str.len().value_counts().sort_index().to_dict()}")

There are 45843 rows and 3 columns in the links dataset


,movieId,imdbId,tmdbId
0,1,0114709,862
1,2,0113497,8844
2,3,0113228,15602
3,4,0114885,31357
4,5,0113041,11862


movieId: 0 values that are not digits, number of digits: {1: 9, 2: 89, 3: 883, 4: 7356, 5: 12128, 6: 25378}
imdbId: 0 values that are not digits, number of digits: {7: 45843}
tmdbId: 0 values that are not digits, number of digits: {1: 4, 2: 59, 3: 733, 4: 2992, 5: 25662, 6: 16174}


In [35]:
# imdbId always has 7 digits with leading zeros, which are dropped when read as a number
# tmdbId has missing values and would be read as float, so it is read as a nullable integer
links = pd.read_csv(root + '/links.csv', dtype={'tmdbId': 'Int64'})
display(links.dtypes)

print(f"Number of duplicate rows: {links.duplicated().sum()}")

movieId    int64
imdbId     int64
tmdbId     Int64
dtype: object

Number of duplicate rows: 0


## movieId

In [36]:
print(f"Nr of rows with no movieId: {links.movieId.isna().sum()}")
print(f"Nr of duplicate movieIds: {links.movieId.duplicated().sum()}")

Nr of rows with no movieId: 0
Nr of duplicate movieIds: 0


## imdbId

In [37]:
print(f"Nr of rows with no imdbId: {links.imdbId.isna().sum()}")
print(f"Nr of duplicate imdbIds: {links.imdbId.duplicated().sum()}")

print(f"imdbId of the first movie in links: {links_raw.imdbId.iloc[0]}")

Nr of rows with no imdbId: 0
Nr of duplicate imdbIds: 0
imdbId of the first movie in links: 0114709


## tmdbId

In [38]:
print(f"There are {links.tmdbId.isna().sum()} ({links.tmdbId.isna().mean():.1%}) rows with no tmdbId")
display(links[links.tmdbId.isna()].head())

# check whether tmdbId is unique
tmdb_vc = links.tmdbId.value_counts()
print(f"There are {links.tmdbId.dropna().duplicated().sum()} duplicate rows, among which the highest number of duplicates for one id is {tmdb_vc.iloc[0]} which occurs {(tmdb_vc == tmdb_vc.iloc[0]).sum()} time")

# the duplicate tmdbIds belong to different movieIds and imdbIds
dup_tmdb = links.tmdbId[links.tmdbId.duplicated(keep=False)].dropna().unique()
display(links[links.tmdbId.isin(dup_tmdb)].sort_values(['tmdbId', 'movieId']).head(6))

There are 219 (0.5%) rows with no tmdbId


,movieId,imdbId,tmdbId
140,142,94878,<NA>
598,604,115978,<NA>
708,720,118114,<NA>
709,721,114103,<NA>
718,730,125877,<NA>


There are 30 duplicate rows, among which the highest number of duplicates for one id is 3 which occurs 1 time


,movieId,imdbId,tmdbId
5905,6003,290538,4912
34144,144606,270288,4912
7388,7587,62229,5511
9215,27136,165303,5511
4148,4241,266860,10991
45197,174533,235679,10991


## movies_metadata

In [39]:
def read_mm():
    with open(root + '/movies_metadata.csv', encoding='utf-8', newline='') as f:
        rows = list(csv.reader(f))

    # merge records that were split by a line break in overview
    fixed = [rows[0]]
    bad_rows = 0 
    for row in rows[1:]:
        if len(fixed[-1]) < len(rows[0]):          # previous row is incomplete
            first_half = fixed.pop()
            row = first_half[:-1] + [first_half[-1] + ' ' + row[0].strip()] + row[1:]
            bad_rows += 1
        fixed.append(row)

    # let pandas read the repaired data
    buf = io.StringIO()
    csv.writer(buf).writerows(fixed)
    buf.seek(0)

    return pd.read_csv(buf)

In [40]:
# read the ids from movies_metadata
mm = read_mm()
mm = mm.dropna(subset=['id']).astype({'id': int}).drop_duplicates('id')
mm['imdbId'] = pd.to_numeric(mm.imdb_id.str[2:], errors='coerce') # remove tt to compare with links

print(f"Nr of movies in movies_metadata that are not in links: {(~mm.id.isin(links.tmdbId)).sum()}")

# tmdbIds in links that are not in movies_metadata, and whether they can be found through imdbId instead
not_in_mm = links[links.tmdbId.notna() & ~links.tmdbId.isin(mm.id)]
print(f"Nr of tmdbIds in links that are not in movies_metadata: {len(not_in_mm)}, of which {not_in_mm.imdbId.isin(mm.imdbId).sum()} can be found by imdbId")
print(f"Nr of rows without tmdbId that can be found by imdbId: {links[links.tmdbId.isna()].imdbId.isin(mm.imdbId).sum()}")

Nr of movies in movies_metadata that are not in links: 0
Nr of tmdbIds in links that are not in movies_metadata: 161, of which 0 can be found by imdbId
Nr of rows without tmdbId that can be found by imdbId: 0


In [41]:
# compare imdbId for the movies that match on tmdbId
joined = links.merge(mm, left_on='tmdbId', right_on='id', suffixes=('', '_mm'))
mismatch = joined[joined.imdbId != joined.imdbId_mm]
print(f"Nr of movies where imdbId differs from movies_metadata: {len(mismatch)}, of which {mismatch.imdbId_mm.isna().sum()} have no imdb_id in movies_metadata")

mismatch = mismatch.dropna(subset=['imdbId_mm'])
print(f"Nr of real mismatches: {len(mismatch)}, of which {mismatch.tmdbId.isin(dup_tmdb).sum()} have a duplicate tmdbId")

# for the duplicate tmdbIds, count how many of the rows have the same imdbId as movies_metadata
matches = joined[joined.tmdbId.isin(dup_tmdb)].assign(match=lambda d: d.imdbId == d.imdbId_mm).groupby('tmdbId')['match'].sum()
print("Nr of rows per duplicate tmdbId that match movies_metadata:")
display(matches.value_counts())

# the remaining mismatches, the tmdbId points to a movie with the same title but another imdbId (e.g. another release year)
display(mismatch[~mismatch.tmdbId.isin(dup_tmdb)][['movieId', 'imdbId', 'tmdbId', 'imdb_id', 'title', 'release_date']].head())

Nr of movies where imdbId differs from movies_metadata: 112, of which 17 have no imdb_id in movies_metadata
Nr of real mismatches: 95, of which 30 have a duplicate tmdbId
Nr of rows per duplicate tmdbId that match movies_metadata:


match
1    29
Name: count, dtype: int64

,movieId,imdbId,tmdbId,imdb_id,title,release_date
707,723,92123,131232,tt0333373,Two Friends,2002-03-20
868,895,105729,79782,tt1684935,Venice,2010-05-25
1081,1115,114472,141210,tt2250194,The Sleepover,2013-10-12
2114,2223,18876,143750,tt2140519,The Farmer's Wife,2012-06-20
2397,2510,134948,26425,tt0123221,Just the Ticket,1999-02-26


## credits

In [ ]:
credit_ids = pd.read_csv(root + '/credits.csv', usecols=['id']).id
keyword_ids = pd.read_csv(root + '/keywords.csv', usecols=['id']).id
print(f"Nr of tmdbIds in links that are not in credits: {(links.tmdbId.notna() & ~links.tmdbId.isin(credit_ids)).sum()}")
print(f"Nr of tmdbIds in links that are not in keywords: {(links.tmdbId.notna() & ~links.tmdbId.isin(keyword_ids)).sum()}")
print(f"Nr of movies in credits that are not in links: {(~credit_ids.drop_duplicates().isin(links.tmdbId)).sum()}")

# the tmdbId that is in movies_metadata, but not in credits
display(mm[mm.id.isin(links.tmdbId) & ~mm.id.isin(credit_ids)][['id', 'title', 'release_date']])

Nr of tmdbIds in links that are not in credits: 162
Nr of movies in credits that are not in links: 0


## ratings

In [43]:
# only movieId is needed
ratings_movies = pd.read_csv(root + '/ratings.csv', usecols=['movieId']).movieId
rated = ratings_movies.drop_duplicates()
print(f"There are {len(rated)} rated movies, of which {(~rated.isin(links.movieId)).sum()} are not in links")
print(f"Nr of movies in links without ratings: {(~links.movieId.isin(rated)).sum()}")

# ratings for movies that cannot be joined with movies_metadata (no tmdbId, or tmdbId not in movies_metadata)
no_metadata = links.loc[~links.tmdbId.isin(mm.id).fillna(False), 'movieId']
print(f"Nr of movies that cannot be joined with movies_metadata: {len(no_metadata)}")
print(f"Nr of ratings for these movies: {ratings_movies.isin(no_metadata).sum()} ({ratings_movies.isin(no_metadata).mean():.2%})")

There are 45115 rated movies, of which 0 are not in links
Nr of movies in links without ratings: 728
Nr of movies that cannot be joined with movies_metadata: 380
Nr of ratings for these movies: 42707 (0.16%)


## Findings and decisions

| Finding | Decision |
|---|---|
|`links.csv` loads cleanly with 45,843 rows and 3 columns (`movieId`, `imdbId`, `tmdbId`), and every value is a whole number|No action needed|
|`imdbId` is  unique. It is always stored as 7 digits with leading zeros, which are dropped when read as a number. There are 112 `imdbId` that does not match with the one in movies_metadate (among which 17 have no im`imdb_id` in movies_metadata). 30 of the 95 real mismatches arise from duplicate `tmdbId`. To join with `imdb_id` in movies_metadata (written as `tt` + 7 digits) the format must be converted. |Store the value as string with tt, to be able to join the information. Drop the rows that are not null in movies_metadata and still mismatches after correcting `tmdbId` 
|`tmdbId` is not unique, has 219 (0.5 %) missing values and is read as float by default. 29 `tmdbId`s occur more than once (30 surplus rows, one id three times), each time with a different `movieId` and `imdbId`. In every case exactly one of the rows has the same imdbId as movies_metadata, so the other rows link to the wrong movie. Further there are 161 `tmdbId` that cannot be found in movies_metadata and 162 that cannot be found in credits. The extra one is School’s out (id 401840), which is in movies_metadata but has no credits or keywords.|Read `tmdbId` as nullable int. Only keep the correct row that links to a movie in movies_metadata. Drop the rows with `tmdbId` that does not exist in movies_metadata|
|`movieId` is unique and have no missing values. It starts on 1 and increments. |No action needed|
|All 45,115 rated movies in ratings.csv are in links, while 728 movies in links (1.6 %) have no ratings.Futher there are 380 movies that cannot be joined with movies_metadata, thus 42,707 ratings are lost|Ratings for movies that are not in the cleaned links are dropped when ratings are mapped to movies. This covers these 380 movies and the 95 links where imdbId does not match movies_metadata|